# Battery Cell Design — Quick Start

This notebook walks through a complete episode of `prodata/CellDesign-v0`.

**What happens:** An agent writes Python code that defines battery cell parameters.
PyBaMM simulates the electrochemistry and returns energy density, cycle life,
peak temperature, and cost. The verifier scores the design multi-dimensionally.

**No API key required.** The basic verifier runs locally.

In [ ]:
# Install (skip if already installed)
# !pip install "prodata[battery]" -q

In [ ]:
import gymnasium as gym
import prodata.battery_gym  # registers environments

env = gym.make("prodata/CellDesign-v0")
obs, info = env.reset(options={"task_id": "cell_ev_standard_001"})

print("Task:", obs["task_description"][:200])
print()
print(f"Target energy density:  {obs['target_energy_density_whkg'][0]:.0f} Wh/kg")
print(f"Required cycle life:    {obs['min_cycle_life'][0]:.0f} cycles")
print(f"Max peak temperature:   {obs['max_peak_temp_c'][0]:.0f} °C")
print(f"Cost budget:            ${obs['max_cost_kwh'][0]:.0f}/kWh")

In [ ]:
# --- Zero-shot design (naive, suboptimal) ---
# This is what a model might output before RL training.
# Round numbers, no understanding of energy-cycle-life tradeoffs.

zero_shot_action = """
params = {
    "chemistry": "NMC532",
    "negative_electrode_thickness": 50e-6,    # m — too thin
    "negative_electrode_porosity": 0.50,      # too high, wastes space
    "negative_electrode_particle_radius": 10e-6,  # too large, bad kinetics
    "positive_electrode_thickness": 50e-6,    # too thin
    "positive_electrode_porosity": 0.50,      # too high
    "positive_electrode_particle_radius": 8e-6,
    "separator_thickness": 30e-6,
    "separator_porosity": 0.50,
    "ambient_temperature_celsius": 25.0,
}
"""

obs0, reward0, term0, trunc0, info0 = env.step(zero_shot_action)

print("=== Zero-shot design ===")
print(f"  Score:          {reward0:.3f}")
print(f"  Passed:         {info0['success']}")
print(f"  Energy density: {obs0['energy_density_whkg'][0]:.1f} Wh/kg")
print(f"  Cycle life:     {obs0['cycle_life_80pct'][0]:.0f}")
print(f"  Peak temp:      {obs0['peak_temperature_c'][0]:.1f} °C")
print(f"  Cost:           ${obs0['estimated_cost_kwh'][0]:.1f}/kWh")
print(f"  Dimension scores: {info0['dimension_scores']}")

In [ ]:
# Reset the episode
obs, info = env.reset(options={"task_id": "cell_ev_standard_001"})

# --- Optimised design (what RL training produces) ---
# Thicker electrodes → more active material → higher energy density
# Lower porosity → denser packing → more energy per volume
# Smaller particles → better kinetics → better cycle life
# Thinner separator → less dead weight

rl_trained_action = """
# NMC cell optimised for energy density + cycle life tradeoff
# Key insight: thicker electrodes increase energy density, but lower porosity
# must compensate to maintain ion transport and cycle life.

params = {
    "chemistry": "NMC532",
    "negative_electrode_thickness": 95e-6,    # m — thicker for more capacity
    "negative_electrode_porosity": 0.28,      # denser packing
    "negative_electrode_particle_radius": 5e-6,  # smaller = better kinetics
    "positive_electrode_thickness": 80e-6,    # thicker positive
    "positive_electrode_porosity": 0.30,      # dense but not restricted
    "positive_electrode_particle_radius": 3.5e-6,  # small particles
    "separator_thickness": 20e-6,             # thinner = less dead weight
    "separator_porosity": 0.45,
    "ambient_temperature_celsius": 25.0,
}
"""

obs1, reward1, term1, trunc1, info1 = env.step(rl_trained_action)

print("=== RL-trained design ===")
print(f"  Score:          {reward1:.3f}")
print(f"  Passed:         {info1['success']}")
print(f"  Energy density: {obs1['energy_density_whkg'][0]:.1f} Wh/kg")
print(f"  Cycle life:     {obs1['cycle_life_80pct'][0]:.0f}")
print(f"  Peak temp:      {obs1['peak_temperature_c'][0]:.1f} °C")
print(f"  Cost:           ${obs1['estimated_cost_kwh'][0]:.1f}/kWh")
print(f"  Dimension scores: {info1['dimension_scores']}")

In [ ]:
# Compare improvement
print("=== Improvement ===")
print(f"  Score:          {reward0:.3f} → {reward1:.3f}  (+{reward1 - reward0:.3f})")
print(f"  Energy density: {obs0['energy_density_whkg'][0]:.0f} → {obs1['energy_density_whkg'][0]:.0f} Wh/kg")
print(f"  Cycle life:     {obs0['cycle_life_80pct'][0]:.0f} → {obs1['cycle_life_80pct'][0]:.0f} cycles")

In [ ]:
# Browse all 30 tasks
from pathlib import Path
import json

tasks_path = Path(prodata.battery_gym.__file__).parent / "tasks" / "cells_basic.json"
tasks = json.loads(tasks_path.read_text())

print(f"Total tasks: {len(tasks)}")
print()
for t in tasks:
    req = t["requirements"]
    print(f"  [{t['difficulty']:6}] {t['task_id']:35}  "
          f"{req['chemistry']:7}  "
          f"{req['target_energy_density_whkg']:>5.0f} Wh/kg  "
          f"{req['min_cycle_life']:>5} cycles")

In [ ]:
# Minimal RL training loop skeleton
# Replace `your_model_generate()` with your LLM / policy

def your_model_generate(obs: dict) -> str:
    """Your model generates cell design parameters."""
    # Stub: returns a fixed design
    return rl_trained_action

env2 = gym.make("prodata/CellDesign-v0")
episode_rewards = []

for episode in range(3):  # run 3 episodes
    obs, info = env2.reset()
    total_reward = 0.0
    terminated = False

    while not terminated:
        action = your_model_generate(obs)
        obs, reward, terminated, truncated, info = env2.step(action)
        total_reward += reward

    episode_rewards.append(total_reward)
    print(f"Episode {episode+1}: total_reward={total_reward:.3f} | "
          f"task={info['task_id']} | passed={info['success']}")

env2.close()